# SmallCNN Phase 7 training sweep
Upload this notebook to Google Colab, select a T4 GPU under **Runtime ? Change runtime type**, then choose **Runtime ? Run all**. Budget about 2 hours 40 minutes of training plus setup and download.

The 19 runs cover error correction disabled (3), watermark strength (10), and shorter payloads (6). They use the project's BCE training recipe. The completed seed 3?4 comparisons are separate.

On the laptop holding the project key, run the following in PowerShell from the project folder. Paste the clipboard into the notebook's hidden prompt.

```powershell
venv\Scripts\python.exe -c "print(open('keys/watermark_key.bin','rb').read().hex(), end='')" | Set-Clipboard
```

Allow Google Drive access when prompted. Each finished run is saved under MyDrive/smallcnn_phase7. On a new GPU runtime, use the same Drive account and Run all to restore completed runs. An interrupted run restarts from epoch 1. The final cell downloads colab_smallcnn_sweeps.zip.

In [ ]:
# 1. GPU check: stop early if Colab gave a CPU-only runtime
import sys
import torch
print(sys.version.split()[0], "| torch", torch.__version__)
assert torch.cuda.is_available(), "No GPU. Reconnect: Select Kernel -> Colab -> New Colab Server -> T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Load the phase6 source without stored training runs.
import os, pathlib, subprocess

REPO_URL = "https://github.com/shield44-project/post-quantum-cryptography.git"
BRANCH = "phase6"
WORK = "/content/pqw_smallcnn"


def sh(cmd):
    # run a shell command, stream its output, stop Run All if it fails
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError("command failed (exit %d): %s" % (p.returncode, cmd))


if not os.path.isdir(WORK + "/.git"):
    sh(f"git clone --depth 1 --branch {BRANCH} --filter=blob:none --no-checkout {REPO_URL} {WORK}")
    sh(f"git -C {WORK} sparse-checkout set --no-cone '/*' '!/runs/'")
    sh(f"git -C {WORK} checkout {BRANCH}")
os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)
sh("git log --oneline -1")
sh("pip install -q 'reedsolo>=1.7'")
os.environ["CIFAR_ROOT"] = "/content/data"
os.environ["PYTHONUNBUFFERED"] = "1"

In [ ]:
# 3. Watermark key: pasted as hex in a hidden prompt, written only to this runtime, fingerprint checked
from getpass import getpass
from embedding.keys import key_fingerprint

KEY_PATH = pathlib.Path("keys/watermark_key.bin")
EXPECTED = "81b57c4b28884f01"            # the key every model in runs/wm_v1 was trained with

if KEY_PATH.exists() and key_fingerprint(KEY_PATH.read_bytes()) == EXPECTED:
    print("key already on this runtime, fingerprint", EXPECTED, "OK")
else:
    key = bytes.fromhex(getpass("Paste the 64 hex characters of keys/watermark_key.bin (hidden): ").strip())
    fp = key_fingerprint(key)
    if fp != EXPECTED:
        raise ValueError("wrong key: fingerprint %s, expected %s. Nothing was written." % (fp, EXPECTED))
    KEY_PATH.parent.mkdir(exist_ok=True)
    KEY_PATH.write_bytes(key)
    os.chmod(KEY_PATH, 0o600)
    del key
    print("key written, fingerprint", fp, "OK")

In [ ]:
# 4. CIFAR-10 to local disk (fails here, not 20 minutes into training, if the download breaks)
import torchvision
for train in (True, False):
    ds = torchvision.datasets.CIFAR10("/content/data", train=train, download=True)
    print("train" if train else "test", len(ds))

In [ ]:
from collections import Counter
from experiments.phase7.run_sweeps import plan

groups = {"ecc_off": 3, "lambda": 10, "payload": 6}
selected = [row for row in plan() if row[0] in groups]
assert Counter(row[0] for row in selected) == groups, "Unexpected sweep plan. Check the repository version."
for group, description, argv, final in selected:
    print(group, description, "done" if final.exists() else "pending")

In [ ]:
from google.colab import drive
import shutil
import shlex
drive.mount("/content/drive")
backup = pathlib.Path("/content/drive/MyDrive/smallcnn_phase7")
names = ("metrics.csv", "model.pt", "final.json")

def complete(folder):
    return all((folder / name).is_file() and (folder / name).stat().st_size > 0 for name in names)

def save_run(source, destination):
    destination.mkdir(parents=True, exist_ok=True)
    for name in names:
        tmp = destination / (name + ".partial")
        shutil.copyfile(source / name, tmp)
        tmp.replace(destination / name)

for i, (group, description, argv, final) in enumerate(selected, 1):
    local = final.parent
    saved = backup / local
    if complete(saved):
        save_run(saved, local)
        print(f"[{i}/19] Restored {description}", flush=True)
        continue
    if not complete(local):
        if final.exists():
            raise RuntimeError(f"Incomplete files alongside {final}. Inspect this run before continuing.")
        print(f"[{i}/19] Training {description}", flush=True)
        sh(shlex.join([sys.executable, "-u", *argv]))
    if not complete(local):
        raise RuntimeError(f"Missing training outputs in {local}")
    save_run(local, saved)
    print(f"[{i}/19] Saved {description} to Drive", flush=True)

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

expected_files = [final.parent / name for _, _, _, final in selected
                  for name in ("final.json", "model.pt", "metrics.csv")]
missing = [str(p) for p in expected_files if not p.is_file() or p.stat().st_size == 0]
if missing:
    raise RuntimeError(f"Training outputs still missing: {missing}")
archive = pathlib.Path("/content/colab_smallcnn_sweeps.zip")
with ZipFile(archive, "w", ZIP_DEFLATED) as bundle:
    for path in expected_files:
        bundle.write(path, path.as_posix())
with ZipFile(archive) as bundle:
    assert len(bundle.namelist()) == 57 and bundle.testzip() is None
print(f"Ready to download {archive.name}: 19 runs, 57 files.")
files.download(str(archive))